# Milestone 5

## Contrastive Pair Margin

$score(i) = \frac{1}{2}\Big(\mathrm{mean}(L(I_i,C_{neg})) - L(I_i,C_i)\Big) + \frac{1}{2}\Big(\mathrm{mean}(L(I_{neg},C_i)) - L(I_i,C_i)\Big)$

In [ ]:
!pip install -q transformers datasets accelerate Pillow scikit-learn tqdm num2words

In [2]:
# imports
import io
import os
import random
import argparse
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm
from transformers import AutoProcessor, SmolVLMForConditionalGeneration

In [3]:
MODEL_ID = "UBC-SLIME/colx_585_vlm"
DATASET_ID = "UBC-SLIME/colx585_group_project_data"

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def load_image(example: Dict) -> Image.Image:
    if isinstance(example["image"], dict) and "bytes" in example["image"]:
        return Image.open(io.BytesIO(example["image"]["bytes"])).convert("RGB")
    return example["image"].convert("RGB")


def split_text_pair(text: str):
    parts = text.split("\n", 1)
    user_text = parts[0]
    assistant_text = parts[1] if len(parts) > 1 else ""
    return user_text, assistant_text


def build_messages(full_text: str):
    user_text, assistant_text = split_text_pair(full_text)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": assistant_text},
            ],
        },
    ]
    return messages


In [5]:
def load_model_and_processor(device: str):
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.image_processor.do_image_splitting = False
    processor.image_processor.size = {"longest_edge": 512}
    processor.image_processor.max_image_size = {"longest_edge": 512}

    model = SmolVLMForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        _attn_implementation="sdpa",
        trust_remote_code=True,
    ).to(device)

    model.eval()
    return model, processor


def prepare_inputs(image: Image.Image, full_text: str, processor, device: str):
    messages = build_messages(full_text)
    text = processor.apply_chat_template(messages, tokenize=False)
    inputs = processor(text=text, images=[image], return_tensors="pt").to(device)

    if device == "cuda":
        inputs = {
            k: v.to(torch.bfloat16) if v.dtype == torch.float32 else v
            for k, v in inputs.items()
        }

    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return inputs, labels

In [6]:
@torch.no_grad()
def compute_pair_loss(image: Image.Image, full_text: str, model, processor, device: str) -> float:
    inputs, labels = prepare_inputs(image, full_text, processor, device)
    outputs = model(**inputs, labels=labels)
    return float(outputs.loss.item())


def load_split(split_name: str, max_examples: Optional[int] = None) -> List[Dict]:
    ds = load_dataset(DATASET_ID, split=split_name)
    records = []

    for example in ds:
        record = {
            "id": example["id"],
            "image": example["image"],
            "text": example["text"],
        }
        if "is_member" in example:
            record["label"] = example["is_member"]
        if "type" in example:
            record["type"] = example["type"]

        records.append(record)

        if max_examples is not None and len(records) >= max_examples:
            break

    return records


def sample_negative_indices(n: int, idx: int, k: int, rng: random.Random) -> List[int]:
    if n <= 1:
        return []

    candidates = list(range(n))
    candidates.remove(idx)

    if k >= len(candidates):
        return candidates

    return rng.sample(candidates, k)

In [7]:
def compute_contrastive_features_for_split(
    records: List[Dict],
    model,
    processor,
    device: str,
    k_caption_neg: int = 5,
    k_image_neg: int = 5,
    seed: int = 42,
) -> pd.DataFrame:
    rng = random.Random(seed)
    n = len(records)
    results = []

    for idx in tqdm(range(n), desc="Computing contrastive features"):
        example = records[idx]

        try:
            image_i = load_image(example)
            text_i = example["text"]

            pos_loss = compute_pair_loss(image_i, text_i, model, processor, device)

            caption_neg_ids = sample_negative_indices(n, idx, k_caption_neg, rng)
            caption_neg_losses = []
            for j in caption_neg_ids:
                text_j = records[j]["text"]
                loss_ij = compute_pair_loss(image_i, text_j, model, processor, device)
                caption_neg_losses.append(loss_ij)

            image_neg_ids = sample_negative_indices(n, idx, k_image_neg, rng)
            image_neg_losses = []
            for j in image_neg_ids:
                image_j = load_image(records[j])
                loss_ji = compute_pair_loss(image_j, text_i, model, processor, device)
                image_neg_losses.append(loss_ji)

            cap_mean = float(np.mean(caption_neg_losses)) if caption_neg_losses else np.nan
            cap_min = float(np.min(caption_neg_losses)) if caption_neg_losses else np.nan
            cap_max = float(np.max(caption_neg_losses)) if caption_neg_losses else np.nan
            cap_std = float(np.std(caption_neg_losses)) if caption_neg_losses else np.nan

            img_mean = float(np.mean(image_neg_losses)) if image_neg_losses else np.nan
            img_min = float(np.min(image_neg_losses)) if image_neg_losses else np.nan
            img_max = float(np.max(image_neg_losses)) if image_neg_losses else np.nan
            img_std = float(np.std(image_neg_losses)) if image_neg_losses else np.nan

            all_neg = caption_neg_losses + image_neg_losses
            all_neg_mean = float(np.mean(all_neg)) if all_neg else np.nan
            all_neg_min = float(np.min(all_neg)) if all_neg else np.nan

            cap_margin_mean = cap_mean - pos_loss if not np.isnan(cap_mean) else np.nan
            img_margin_mean = img_mean - pos_loss if not np.isnan(img_mean) else np.nan
            combined_margin_mean = all_neg_mean - pos_loss if not np.isnan(all_neg_mean) else np.nan

            cap_margin_hard = cap_min - pos_loss if not np.isnan(cap_min) else np.nan
            img_margin_hard = img_min - pos_loss if not np.isnan(img_min) else np.nan
            combined_margin_hard = all_neg_min - pos_loss if not np.isnan(all_neg_min) else np.nan

            #new feature
            all_losses = caption_neg_losses + image_neg_losses + [pos_loss]
            rank = sum(l <= pos_loss for l in all_losses) / len(all_losses)
            rank_score = 1 - rank
            ratio_mean = pos_loss / (1e-6 + all_neg_mean) if not np.isnan(all_neg_mean) else np.nan
            ratio_hard = pos_loss / (1e-6 + all_neg_min) if not np.isnan(all_neg_min) else np.nan
            #
            score = 0.5 * cap_margin_hard + 0.5 * img_margin_hard

            row = {
                "id": example["id"],
                "pos_loss": pos_loss,
                "cap_neg_mean": cap_mean,
                "cap_neg_min": cap_min,
                "cap_neg_max": cap_max,
                "cap_neg_std": cap_std,
                "img_neg_mean": img_mean,
                "img_neg_min": img_min,
                "img_neg_max": img_max,
                "img_neg_std": img_std,
                "cap_margin_mean": cap_margin_mean,
                "img_margin_mean": img_margin_mean,
                "combined_margin_mean": combined_margin_mean,
                "cap_margin_hard": cap_margin_hard,
                "img_margin_hard": img_margin_hard,
                "combined_margin_hard": combined_margin_hard,
                "score": score,
                "rank_score": rank_score,
                "ratio_mean": ratio_mean,
                "ratio_hard": ratio_hard,
            }

            if "label" in example:
                row["label"] = example["label"]
            if "type" in example:
                row["type"] = example["type"]

            results.append(row)

        except Exception as e:
            print(f"Skipped {example['id']} due to error: {e}")

    return pd.DataFrame(results)


In [8]:
def tpr_at_fpr(y_true, y_score, target_fpr: float = 0.1) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    valid = np.where(fpr <= target_fpr)[0]
    if len(valid) == 0:
        return 0.0
    return float(np.max(tpr[valid]))


def evaluate(df: pd.DataFrame, split_name: str):
    if "label" not in df.columns:
        print(f"{split_name}: no labels available.")
        return

    y_true = df["label"].values
    y_score = df["score"].values

    auc = roc_auc_score(y_true, y_score)
    tpr01 = tpr_at_fpr(y_true, y_score, target_fpr=0.1)

    print(f"[{split_name}] AUC = {auc:.4f} | TPR@FPR=0.1 = {tpr01:.4f}")

### Small Try First

In [ ]:
pip install num2words

In [ ]:
set_seed(42)

device = get_device()
print("Using device:", device)

model, processor = load_model_and_processor(device)

In [ ]:
val_records = load_split("validation", max_examples=10)
print("Validation examples loaded:", len(val_records))

print(val_records[0].keys())
print(val_records[0]["text"][:300])

In [ ]:
df_val_debug = compute_contrastive_features_for_split(
    val_records,
    model,
    processor,
    device,
    k_caption_neg=8,
    k_image_neg=8,
    seed=24,
)

In [ ]:
df_val_debug.head()

In [ ]:
evaluate(df_val_debug, "validation_debug")

In [ ]:
df_val_debug["score"].describe()

In [ ]:
# save debud results
df_val_debug.to_csv("val_contrastive_features_debug.csv", index=False)

In [ ]:
# bigger validation subset
val_records_50 = load_split("validation", max_examples=50)

df_val_50 = compute_contrastive_features_for_split(
    val_records_50,
    model,
    processor,
    device,
    k_caption_neg=2,
    k_image_neg=2,
    seed=42,
)

evaluate(df_val_50, "validation_50")
df_val_50.to_csv("val_contrastive_features_50.csv", index=False)

In [ ]:
val_records_50 = load_split("validation", max_examples=500)

df_val_50_k5 = compute_contrastive_features_for_split(
    val_records_50,
    model,
    processor,
    device,
    k_caption_neg=5,
    k_image_neg=5,
    seed=42,
)

evaluate(df_val_50_k5, "validation_50_k5")
df_val_50_k5.to_csv("val_contrastive_features_50_k5.csv", index=False)

In [ ]:
test_records = load_split("test")

df_test = compute_contrastive_features_for_split(
    test_records,
    model,
    processor,
    device,
    k_caption_neg=2,
    k_image_neg=2,
    seed=42,
)

df_test[["id", "score"]].to_csv("submission_contrastive_debug.csv", index=False)
df_test.head()

In [ ]:
df_test.rename(columns={"score": "is_member"})[["id", "is_member"]].to_csv(
    "submission_contrastive.csv", index=False
)

In [ ]:
from google.colab import files
files.download("submission_contrastive.csv")

## New Approach - Classifier Method

In [9]:
set_seed(24)

device = get_device()
print("Using device:", device)

model, processor = load_model_and_processor(device)

Using device: cuda


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

In [10]:
# train features on train split
train_records = load_split("train")
print("Train examples loaded:", len(train_records))

df_train = compute_contrastive_features_for_split(
    train_records,
    model,
    processor,
    device,
    k_caption_neg=10,
    k_image_neg=10,
    seed=24,
)

df_train.head()

train.parquet:   0%|          | 0.00/334M [00:00<?, ?B/s]

dev.parquet:   0%|          | 0.00/67.0M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/338M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Train examples loaded: 6000


Computing contrastive features: 100%|██████████| 6000/6000 [5:51:59<00:00,  3.52s/it]  


,id,pos_loss,cap_neg_mean,cap_neg_min,cap_neg_max,cap_neg_std,img_neg_mean,img_neg_min,img_neg_max,img_neg_std,...,combined_margin_mean,cap_margin_hard,img_margin_hard,combined_margin_hard,score,rank_score,ratio_mean,ratio_hard,label,type
0,cauldron_ln_15810,0.698831,0.542172,0.359085,0.830848,0.154441,0.701428,0.698711,0.704133,0.001871,...,-0.077032,-0.339746,-0.000120,-0.339746,-0.169933,0.523810,1.123883,1.946137,0,unseen_both
1,cauldron_ln_24846,0.758338,0.525224,0.205038,0.931585,0.197915,0.755237,0.751339,0.758553,0.002083,...,-0.118108,-0.553300,-0.007000,-0.553300,-0.280150,0.095238,1.184475,3.698503,0,unseen_both
2,mix_SI_UT_0_19,0.354725,0.410863,0.183650,0.679865,0.156481,0.348498,0.341245,0.357198,0.005632,...,0.024956,-0.171075,-0.013480,-0.171075,-0.092278,0.428571,0.934269,1.931520,0,seen_img_unseen_txt
3,mix_SI_UT_0_896,0.407683,0.496280,0.306305,0.748489,0.135182,0.406869,0.403681,0.409827,0.001943,...,0.043891,-0.101378,-0.004003,-0.101378,-0.052691,0.476190,0.902803,1.330968,0,seen_img_unseen_txt
4,cauldron_ln_138,0.334805,0.481678,0.203217,0.722496,0.162825,0.339730,0.335034,0.343149,0.002253,...,0.075899,-0.131588,0.000229,-0.131588,-0.065680,0.857143,0.815196,1.647518,1,seen_both


In [11]:
# Validation features
val_records = load_split("validation")
print("Validation examples loaded:", len(val_records))

df_val = compute_contrastive_features_for_split(
    val_records,
    model,
    processor,
    device,
    k_caption_neg=10,
    k_image_neg=10,
    seed=24,
)

df_val.head()

Validation examples loaded: 1200


Computing contrastive features: 100%|██████████| 1200/1200 [1:10:44<00:00,  3.54s/it]


,id,pos_loss,cap_neg_mean,cap_neg_min,cap_neg_max,cap_neg_std,img_neg_mean,img_neg_min,img_neg_max,img_neg_std,...,combined_margin_mean,cap_margin_hard,img_margin_hard,combined_margin_hard,score,rank_score,ratio_mean,ratio_hard,label,type
0,cauldron_ln_18258,0.629163,0.505526,0.187105,0.962647,0.207555,0.629402,0.624242,0.635693,0.002861,...,-0.061699,-0.442058,-0.004921,-0.442058,-0.223489,0.333333,1.108725,3.362604,0,unseen_both
1,mix_SI_UT_4500_263,0.178469,0.495628,0.289433,0.782216,0.138226,0.180169,0.176966,0.185460,0.002170,...,0.159430,0.110965,-0.001502,-0.001502,0.054731,0.857143,0.528171,1.008483,0,seen_img_unseen_txt
2,cauldron_ln_27714,0.659445,0.435611,0.176686,0.948821,0.218824,0.659433,0.653563,0.673740,0.005399,...,-0.111923,-0.482759,-0.005882,-0.482759,-0.244320,0.238095,1.204415,3.732286,0,unseen_both
3,mix_UI_ST_4500_189,0.802297,0.615955,0.426112,1.069955,0.185405,0.810126,0.805310,0.819786,0.004527,...,-0.089257,-0.376186,0.003012,-0.376186,-0.186587,0.523810,1.125176,1.882829,0,unseen_img_seen_txt
4,cauldron_ln_21048,0.303821,0.557156,0.180566,1.237770,0.322917,0.308786,0.303109,0.315053,0.003654,...,0.129150,-0.123255,-0.000713,-0.123255,-0.061984,0.857143,0.701711,1.682596,1,seen_both


In [13]:
# Choose features
feature_cols = [
    "pos_loss",
    "cap_margin_mean",
    "img_margin_mean",
    "combined_margin_mean",
    "cap_margin_hard",
    "img_margin_hard",
    "combined_margin_hard",
    "rank_score",
    "ratio_mean",
    "ratio_hard",
]

In [14]:
# Clean
df_train[feature_cols].isna().sum()
df_val[feature_cols].isna().sum()
df_train[feature_cols] = df_train[feature_cols].fillna(0)
df_val[feature_cols] = df_val[feature_cols].fillna(0)

In [15]:
# Train logistic regression
from sklearn.linear_model import LogisticRegression

X_train = df_train[feature_cols]
y_train = df_train["label"]

X_val = df_val[feature_cols]
y_val = df_val["label"]

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

In [16]:
# Predict on validation
df_val["clf_score"] = clf.predict_proba(X_val)[:, 1]
df_val[["id", "label", "clf_score"]].head()

,id,label,clf_score
0,cauldron_ln_18258,0,0.217585
1,mix_SI_UT_4500_263,0,0.273216
2,cauldron_ln_27714,0,0.207638
3,mix_UI_ST_4500_189,0,0.285861
4,cauldron_ln_21048,1,0.325852


In [17]:
# Evaluation
auc = roc_auc_score(y_val, df_val["clf_score"])
tpr01 = tpr_at_fpr(y_val, df_val["clf_score"], target_fpr=0.1)

print(f"[validation_classifier] AUC = {auc:.4f} | TPR@FPR=0.1 = {tpr01:.4f}")

[validation_classifier] AUC = 0.6043 | TPR@FPR=0.1 = 0.1867


In [19]:
# Test features
test_records = load_split("test")  
print("Test examples loaded:", len(test_records))

df_test = compute_contrastive_features_for_split(
    test_records,
    model,
    processor,
    device,
    k_caption_neg=5,
    k_image_neg=5,
    seed=42,
)

Test examples loaded: 6000


Computing contrastive features: 100%|██████████| 6000/6000 [3:07:08<00:00,  1.87s/it]  


In [26]:
# Clean
df_test[feature_cols] = df_test[feature_cols].fillna(0)
df_test["is_member"] = clf.predict_proba(df_test[feature_cols])[:, 1]

In [27]:
submission = df_test[["id", "is_member"]].copy()

#submission.to_csv("submission_classifier.csv", index=False)

submission.head()

,id,is_member
0,cauldron_ln_8044,0.306413
1,cauldron_ln_7863,0.117164
2,mix_SI_UT_5400_19,0.200916
3,mix_SI_UT_5400_896,0.188279
4,cauldron_ln_17842,0.155974


In [28]:
submission.to_csv("/kaggle/working/submission_classifier.csv", index=False)